In [ ]:
import pandas as pd
import numpy as np
import re

# 1. Загружаем исходный файл
df = pd.read_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\raw_trades.csv')

# 2. Чистка чисел (убираем всё, кроме цифр и точек)
def clean_money(value):
    if pd.isna(value): return value
    # Оставляем только цифры, точку и минус (для профита)
    cleaned = re.sub(r'[^\d\.\-]', '', str(value))
    return cleaned

# Применяем чистку к ценам и профиту
money_cols = ['Average Entry Price', 'Exit Price', 'Profit %']
for col in money_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].apply(clean_money), errors='coerce')

# 3. Чистка тикеров (удаляем скобки с содержимым и звезды)
df['Symbol'] = df['Symbol'].apply(lambda x: re.sub(r'\(.*?\)', '', str(x)).replace('*', '').strip().upper() if pd.notna(x) else x)

# 4. Удаляем налоги и мусор по ключевым словам
keywords_to_remove = ['tax', 'fee', 'commission', 'interest', 'adjustment', 'div']
df = df[~df['Symbol'].str.contains('|'.join(keywords_to_remove), case=False, na=False)]
df = df.dropna(subset=['Symbol'])

# 5. Даты и Длительность
df['Entry Date'] = pd.to_datetime(df['Entry Date'], errors='coerce')
df['Exit Date'] = pd.to_datetime(df['Exit Date'], errors='coerce')
df['Trade Duration'] = (df['Exit Date'] - df['Entry Date']).dt.days

# 6. Временные колонки (год, месяц, день недели)
df['Year'] = df['Entry Date'].dt.year.astype('Int64')
df['Month'] = df['Entry Date'].dt.month.astype('Int64')
df['Weekday'] = df['Entry Date'].dt.day_name()

# 7. Восстановление оставшихся пустот (если они были в оригинале)
def final_recovery(row):
    entry, exit_p, profit = row['Average Entry Price'], row['Exit Price'], row['Profit %']
    direction = 1 if row['Long/Short'] == 'Long' else -1
    
    if pd.isna(entry) and pd.notna(exit_p) and pd.notna(profit):
        entry = exit_p / (1 + (profit / 100) * direction)
    if pd.isna(exit_p) and pd.notna(entry) and pd.notna(profit):
        exit_p = entry * (1 + (profit / 100) * direction)
    if pd.isna(profit) and pd.notna(entry) and pd.notna(exit_p) and entry != 0:
        profit = ((exit_p - entry) / entry) * 100 * direction
    return pd.Series([entry, exit_p, profit])

df[['Average Entry Price', 'Exit Price', 'Profit %']] = df.apply(final_recovery, axis=1)

# Удаляем пустую колонку, если она есть
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# Итоговый отчет
print(f"Обработано строк: {len(df)}")
print(f"Осталось пропусков в цене входа: {df['Average Entry Price'].isnull().sum()}")
df.to_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\trades_fixed.csv', index=False)
print("✅ Файл 'trades_fixed.csv' готов. Теперь цены на месте!")

Обработано строк: 2171
Осталось пропусков в цене входа: 1
✅ Файл 'trades_final_fixed.csv' готов. Теперь цены на месте!


In [4]:


# Загружаем наш последний файл (где мы уже восстановили всё, что удалось)
df = pd.read_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\trades_fixed.csv')

# 1. Считаем пустые ячейки в ключевых колонках
# Нам критически важны: Symbol, Entry Date, Average Entry Price, Exit Price, Profit %
critical_cols = ['Symbol', 'Entry Date', 'Average Entry Price', 'Exit Price', 'Profit %']

print("--- Отчет о пустых ячейках перед удалением ---")
missing_report = df[critical_cols].isnull().sum()
print(missing_report[missing_report > 0])

# 2. Считаем, сколько всего "битых" строк (где есть хотя бы один пропуск в важных колонках)
rows_with_nan = df[critical_cols].isnull().any(axis=1).sum()
print(f"\nВсего строк с пропусками в ключевых данных: {rows_with_nan}")

# 3. Удаляем эти строки
# Мы сохраняем только те строки, где во всех критических колонках есть данные
df_cleaned = df.dropna(subset=critical_cols)

# 4. Итоговый расчет
final_count = len(df_cleaned)
print(f"Удалено строк: {rows_with_nan}")
print(f"Осталось чистых строк для анализа: {final_count}")

# 5. Сохраняем финальный результат
df_cleaned.to_csv(r'C:\Users\USER\Desktop\Folder\Murat\DS\Outpeer\Capstone\NewTry\trades_dataset.csv', index=False)

print("\n✅ Файл 'trades_dataset.csv' готов к анализу!")

--- Отчет о пустых ячейках перед удалением ---
Entry Date             3
Average Entry Price    1
Exit Price             1
Profit %               1
dtype: int64

Всего строк с пропусками в ключевых данных: 3
Удалено строк: 3
Осталось чистых строк для анализа: 2168

✅ Файл 'trades_dataset.csv' готов к анализу!
